# Projet 1 — Prévision de consommation électrique

## Prévoir la consommation horaire à horizon 24 h

Ce TP traite une régression multi-horizon : prévoir chacune des 24 prochaines heures à partir de l’historique disponible. On compare les historiques de 24 h, 48 h et 7 jours, les variables calendaires, des baselines, LSTM, TCN et SARIMA.

**Source :** UCI Individual Household Electric Power Consumption, un foyer à Sceaux, décembre 2006 à novembre 2010, mesures à la minute. La cible est Global_active_power, agrégée en puissance moyenne horaire (kW).

**Limites de la source :** aucune température ni météo n’est fournie ; les vacances scolaires non plus. Les jours fériés français sont calculés à partir du calendrier. Les résultats d’un seul foyer ne se généralisent pas automatiquement à un réseau électrique.

## 0 — Environnement et protocole

Exécuter les cellules dans l’ordre depuis la racine du dépôt. Si les données sont absentes, le script du projet les télécharge. Les modèles avancés sont désactivés au départ pour inspecter d’abord les données et les baselines ; activer les options dans la cellule ci-dessous pour les entraîner.

Découpage chronologique 70/15/15 : entraînement, validation, test. Une origine est évaluée toutes les 24 h pour éviter le chevauchement des fenêtres cibles. MAE et RMSE sont exprimées en kW. MAPE est calculée sur les cibles non nulles et s’interprète avec prudence lorsque la consommation est proche de zéro.

In [ ]:
from pathlib import Path
import subprocess, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
HORIZON = 24
LOOKBACKS = [24, 48, 168]
TRAIN_FRAC, VALID_FRAC = .70, .15
MIN_MINUTES = 45
ORIGIN_STRIDE = 24
RUN_NEURAL = False
RUN_CALENDAR_ABLATION = False
RUN_SARIMA = False

np.random.seed(SEED)
sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)
roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in roots if (p/"requirements.txt").exists()), Path.cwd())
DATA = ROOT/"data"/"raw"/"household_power_consumption.txt"
if not DATA.exists():
    subprocess.run([sys.executable, str(ROOT/"scripts"/"download_data.py")], check=True)
print("Projet :", ROOT, "| fichier :", DATA)

## 1 — Charger et agréger les mesures

Une ligne étiquetée 10:00 représente l’intervalle [09:00, 10:00) ; elle est connue à 10:00. À cette origine, les cibles sont les heures 11:00 à 10:00 le lendemain. Une heure ayant moins de 45 mesures valides est marquée absente. Les cibles manquantes ne sont jamais interpolées.

In [ ]:
FEATURES = [
    "Global_active_power", "Global_reactive_power", "Voltage", "Global_intensity",
    "Sub_metering_1", "Sub_metering_2", "Sub_metering_3",
]
SUBMETERS = ["Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]
raw = pd.read_csv(DATA, sep=";", na_values=["?", ""], low_memory=False)
raw["timestamp"] = pd.to_datetime(
    raw["Date"].astype(str)+" "+raw["Time"].astype(str),
    format="%d/%m/%Y %H:%M:%S", errors="coerce"
)
for c in FEATURES:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")
raw = raw.dropna(subset=["timestamp"]).sort_values("timestamp").set_index("timestamp")
duplicates = int(raw.index.duplicated().sum())
raw = raw.loc[~raw.index.duplicated(keep="first")]
display(pd.DataFrame({
    "valeur":[f"{len(raw):,}",str(raw.index.min()),str(raw.index.max()),"1 minute",duplicates],
}, index=["Mesures","Début","Fin","Fréquence","Doublons retirés"]))
display(pd.DataFrame({
    "type":raw[FEATURES].dtypes.astype(str),
    "manquants":raw[FEATURES].isna().sum(),
    "manquants_%":(100*raw[FEATURES].isna().mean()).round(3),
}, index=FEATURES))

minute = raw[FEATURES]
hourly = minute.resample("h",closed="left",label="right").mean()
counts = minute.resample("h",closed="left",label="right").count()
for c in SUBMETERS:
    hourly[c] = minute[c].resample("h",closed="left",label="right").sum(min_count=1)
hourly, counts = hourly.asfreq("h"), counts.asfreq("h")
for c in FEATURES:
    hourly.loc[counts[c] < MIN_MINUTES,c] = np.nan
target = hourly["Global_active_power"].rename("target_kw")
y, time_index = target.to_numpy(float), hourly.index
n = len(hourly)
train_end, valid_end = int(n*TRAIN_FRAC), int(n*(TRAIN_FRAC+VALID_FRAC))
display(pd.DataFrame({"valeur":[f"{n:,}",str(time_index.min()),str(time_index.max()),
    f"{target.notna().sum():,}",f"{100*target.isna().mean():.2f}%"]},
    index=["Grille horaire","Début","Fin","Cibles valides","Taux cible manquante"]))
print("Borne entraînement :",time_index[train_end],"| borne validation :",time_index[valid_end])

## 2 — Séparation temporelle

Les fenêtres sont classées selon leurs 24 cibles : elles doivent rester intégralement dans leur partition. Les historiques peuvent traverser une borne s’ils seraient connus au moment de la prévision. Chaque fenêtre doit aussi avoir 24 cibles observées.

In [ ]:
origins_all = np.arange(max(LOOKBACKS)-1,n-HORIZON,ORIGIN_STRIDE)
def complete_origins(origins):
    return np.asarray([i for i in origins if np.isfinite(y[i+1:i+HORIZON+1]).all()],dtype=int)
train_origins = complete_origins(origins_all[origins_all+HORIZON < train_end])
valid_origins = complete_origins(origins_all[(origins_all+1>=train_end)&(origins_all+HORIZON<valid_end)])
test_origins = complete_origins(origins_all[origins_all+1>=valid_end])
trainval_origins = complete_origins(origins_all[origins_all+HORIZON<valid_end])
display(pd.DataFrame([
    {"partition":"Entraînement","origines":len(train_origins)},
    {"partition":"Validation","origines":len(valid_origins)},
    {"partition":"Test","origines":len(test_origins)},
]))
assert min(len(train_origins),len(valid_origins),len(test_origins)) > 0

## 3 — Explorer tendance, cycles et corrélations

Les graphiques sont limités à l’entraînement pour guider les choix sans consulter le test. Les mesures UCI permettent des corrélations entre variables électriques, mais pas avec la météo. Les corrélations contemporaines sont descriptives : elles ne prouvent pas qu’une variable est disponible pour les 24 prochaines heures.

In [ ]:
train = target.iloc[:train_end]
fig,ax = plt.subplots(2,2,figsize=(14,8))
train.resample("D").mean().plot(ax=ax[0,0],title="Moyenne journalière")
train.iloc[:24*14].plot(ax=ax[0,1],title="Deux semaines horaires")
train.dropna().plot.hist(bins=50,ax=ax[1,0],title="Distribution horaire")
train.groupby(train.index.hour).mean().plot(ax=ax[1,1],marker="o",title="Profil moyen par heure")
ax[1,1].set_ylabel("kW")
plt.tight_layout(); plt.show()

weekday = train.groupby(train.index.dayofweek).mean()
weekday.index = ["Lun","Mar","Mer","Jeu","Ven","Sam","Dim"]
display(weekday.rename("Puissance moyenne kW").to_frame().round(3))
daily = train.resample("D").mean().interpolate(method="time",limit=2).dropna()
if len(daily) >= 21:
    from statsmodels.tsa.seasonal import seasonal_decompose
    seasonal_decompose(daily,model="additive",period=7).plot()
    plt.suptitle("Décomposition hebdomadaire de la moyenne journalière",y=1.02)
    plt.tight_layout(); plt.show()

plt.figure(figsize=(9,7))
sns.heatmap(hourly.iloc[:train_end][FEATURES].corr(),annot=True,fmt=".2f",cmap="vlag",center=0)
plt.title("Corrélations contemporaines électriques — entraînement")
plt.tight_layout(); plt.show()
print("Météo absente du fichier UCI ; apparier une source horaire avant toute analyse météo.")

## 4 — Features calendaires

Heure, jour de semaine, saison annuelle, week-end et jours fériés sont connus pour les 24 heures futures. Les sinus/cosinus représentent les cycles sans saut artificiel entre la fin et le début d’un cycle. Les vacances scolaires demandent un calendrier externe.

In [ ]:
CAL_COLS = ["hour_sin","hour_cos","weekday_sin","weekday_cos",
            "year_sin","year_cos","weekend","fr_holiday"]
def calendar_features(idx):
    idx = pd.DatetimeIndex(idx)
    h, wd, doy = idx.hour.to_numpy(), idx.dayofweek.to_numpy(), idx.dayofyear.to_numpy()
    days = np.where(idx.is_leap_year,366.,365.)
    out = pd.DataFrame(index=idx)
    out["hour_sin"],out["hour_cos"] = np.sin(2*np.pi*h/24),np.cos(2*np.pi*h/24)
    out["weekday_sin"],out["weekday_cos"] = np.sin(2*np.pi*wd/7),np.cos(2*np.pi*wd/7)
    out["year_sin"],out["year_cos"] = np.sin(2*np.pi*(doy-1)/days),np.cos(2*np.pi*(doy-1)/days)
    out["weekend"] = (wd>=5).astype(float)
    try:
        import holidays
        fr = holidays.country_holidays("FR",years=range(idx.min().year-1,idx.max().year+2))
        out["fr_holiday"] = [float(t.date() in fr) for t in idx]
    except ImportError:
        out["fr_holiday"] = 0.
    return out[CAL_COLS].astype(float)
cal = calendar_features(time_index)
cal_array = cal.to_numpy(np.float32)
display(cal.head())

## 5 — Baselines et métriques

La persistance répète la dernière mesure. Les références saisonnières reprennent la même heure du jour ou de la semaine précédente. Une valeur historique manquante est remplacée par la médiane de l’entraînement. Tous les modèles sont comparés sur les mêmes cibles de validation.

In [ ]:
def truth(origins):
    return np.stack([y[i+1:i+HORIZON+1] for i in origins])
def score(actual,pred):
    a,p = np.asarray(actual).ravel(),np.asarray(pred).ravel()
    ok = np.isfinite(a)&np.isfinite(p); a,p=a[ok],p[ok]
    nz = np.abs(a)>1e-6
    return {"MAE_kW":mean_absolute_error(a,p),
            "RMSE_kW":np.sqrt(mean_squared_error(a,p)),
            "MAPE_nonzero_pct":100*np.mean(np.abs((a[nz]-p[nz])/a[nz])) if nz.any() else np.nan,
            "n":len(a),"n_MAPE":int(nz.sum())}
train_median = float(np.nanmedian(y[:train_end]))
def baseline(name,origins,values=y,fallback=train_median):
    h = np.arange(1,HORIZON+1)
    if name=="Persistance":
        pred=np.repeat(values[np.asarray(origins),None],HORIZON,axis=1)
    else:
        lag=24 if name=="Saisonnier quotidien" else 168
        ix=np.clip(np.asarray(origins)[:,None]+h[None,:]-lag,0,len(values)-1)
        pred=values[ix].astype(float)
    pred[~np.isfinite(pred)]=fallback
    return np.maximum(pred,0)
BASELINES=["Persistance","Saisonnier quotidien","Saisonnier hebdomadaire"]
valid_truth=truth(valid_origins)
val_predictions={k:baseline(k,valid_origins) for k in BASELINES}
display(pd.DataFrame({k:score(valid_truth,v) for k,v in val_predictions.items()}).T.sort_values("MAE_kW").round(4))

## 6 — Prétraitement sans fuite et fenêtres multi-horizons

L’imputation et la standardisation sont ajustées sur l’entraînement seulement. Les cibles absentes ne sont pas imputées. Le réseau reçoit l’historique électrique ; l’ablation peut ajouter ou retirer le calendrier historique et futur.

In [ ]:
def preprocess(fit_end):
    imputer=SimpleImputer(strategy="median").fit(hourly[FEATURES].iloc[:fit_end])
    filled=imputer.transform(hourly[FEATURES])
    scaler=StandardScaler().fit(filled[:fit_end])
    xs=scaler.transform(filled).astype(np.float32)
    mean=float(np.nanmean(y[:fit_end]))
    std=float(np.nanstd(y[:fit_end]))
    return xs,mean,(std if std>0 else 1.)
def windows(origins,lookback,xs,mean,std,use_calendar=True):
    xh,xf,yy=[],[],[]
    for i in origins:
        hist=xs[i-lookback+1:i+1]
        if use_calendar:
            hist=np.column_stack([hist,cal_array[i-lookback+1:i+1]])
            future=cal_array[i+1:i+HORIZON+1]
        else:
            future=np.zeros((HORIZON,len(CAL_COLS)),np.float32)
        xh.append(hist); xf.append(future); yy.append((y[i+1:i+HORIZON+1]-mean)/std)
    return tuple(np.asarray(v,np.float32) for v in (xh,xf,yy))
xs_train,mean_train,std_train=preprocess(train_end)

## 7 — Modèles LSTM et TCN

Les deux réseaux prédisent directement les 24 valeurs futures. Le benchmark entraîne chaque architecture avec 24 h, 48 h et 168 h d’historique. Il est désactivé par défaut pour laisser le notebook inspecter les données sans lancer d’entraînement ; passer RUN_NEURAL à True pour obtenir les scores.

In [ ]:
def neural_classes():
    import torch
    from torch import nn
    from torch.utils.data import DataLoader,TensorDataset
    class LSTM(nn.Module):
        def __init__(self,d,f):
            super().__init__(); self.rnn=nn.LSTM(d,32,batch_first=True)
            self.head=nn.Sequential(nn.Linear(32+HORIZON*f,64),nn.ReLU(),nn.Linear(64,HORIZON))
        def forward(self,x,f):
            z,_=self.rnn(x); return self.head(torch.cat([z[:,-1],f.flatten(1)],1))
    class Block(nn.Module):
        def __init__(self,ci,co,dil):
            super().__init__(); pad=2*dil
            self.a=nn.Conv1d(ci,co,3,padding=pad,dilation=dil)
            self.b=nn.Conv1d(co,co,3,padding=pad,dilation=dil)
            self.skip=nn.Conv1d(ci,co,1) if ci!=co else nn.Identity()
        def forward(self,x):
            z=torch.relu(self.a(x)[:,:,:x.shape[-1]])
            z=torch.relu(self.b(z)[:,:,:x.shape[-1]])
            return torch.relu(z+self.skip(x))
    class TCN(nn.Module):
        def __init__(self,d,f):
            super().__init__(); self.conv=nn.Sequential(Block(d,32,1),Block(32,32,2),Block(32,32,4))
            self.head=nn.Sequential(nn.Linear(32+HORIZON*f,64),nn.ReLU(),nn.Linear(64,HORIZON))
        def forward(self,x,f):
            z=self.conv(x.transpose(1,2))[:,:,-1]
            return self.head(torch.cat([z,f.flatten(1)],1))
    return torch,nn,DataLoader,TensorDataset,LSTM,TCN

def fit_net(kind,lookback,origins,xs,mean,std,use_calendar=True,epochs=8):
    torch,nn,Loader,Dataset,LSTM,TCN=neural_classes(); torch.manual_seed(SEED)
    x,f,labels=windows(origins,lookback,xs,mean,std,use_calendar)
    model=(LSTM if kind=="LSTM" else TCN)(x.shape[-1],f.shape[-1])
    ds=Dataset(torch.tensor(x),torch.tensor(f),torch.tensor(labels))
    loader=Loader(ds,batch_size=128,shuffle=True,generator=torch.Generator().manual_seed(SEED))
    opt=torch.optim.Adam(model.parameters(),lr=1e-3); loss_fn=nn.MSELoss()
    for _ in range(epochs):
        for xb,fb,yb in loader:
            opt.zero_grad(); loss=loss_fn(model(xb,fb),yb); loss.backward(); opt.step()
    return model
def predict_net(model,lookback,origins,xs,mean,std,use_calendar=True):
    torch,*_=neural_classes(); x,f,_=windows(origins,lookback,xs,mean,std,use_calendar)
    model.eval()
    with torch.no_grad(): pred=model(torch.tensor(x),torch.tensor(f)).numpy()
    return np.maximum(pred*std+mean,0)

neural_val,neural_models,neural_specs={},{},{}
if RUN_NEURAL:
    for kind in ["LSTM","TCN"]:
        for lookback in LOOKBACKS:
            name=f"{kind} | {lookback} h | calendrier"
            model=fit_net(kind,lookback,train_origins,xs_train,mean_train,std_train)
            neural_models[name]=model
            neural_specs[name]={"kind":kind,"lookback":lookback,"calendar":True}
            neural_val[name]=predict_net(model,lookback,valid_origins,xs_train,mean_train,std_train)
    display(pd.DataFrame({k:score(valid_truth,v) for k,v in neural_val.items()}).T.sort_values("MAE_kW").round(4))
else:
    print("Réseaux désactivés : activer RUN_NEURAL et réexécuter cette cellule.")

## 8 — SARIMA et ablation calendrier (options)

SARIMA est ajusté sur les 120 derniers jours disponibles avec une saisonnalité quotidienne, prévoit 24 h à chaque origine, puis met à jour son état avec les observations mesurées. Les paramètres restent fixes pendant validation. L’ablation compare la meilleure configuration neuronale avec et sans calendrier.

In [ ]:
def sarima_predict(fit_end,origins,fit_days=120):
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    start=max(0,fit_end-fit_days*24)
    series=pd.Series(y[start:fit_end],index=time_index[start:fit_end])
    fitted=SARIMAX(series,order=(1,0,1),seasonal_order=(1,0,1,24),
        enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=40)
    last=fit_end-1; output=[]
    for origin in origins:
        if origin>last:
            upd=pd.Series(y[last+1:origin+1],index=time_index[last+1:origin+1])
            fitted=fitted.append(upd,refit=False); last=origin
        output.append(np.maximum(np.asarray(fitted.get_forecast(HORIZON).predicted_mean),0))
        stop=origin+HORIZON+1
        upd=pd.Series(y[origin+1:stop],index=time_index[origin+1:stop])
        fitted=fitted.append(upd,refit=False); last=stop-1
    return np.stack(output)

if RUN_SARIMA:
    sarima_val=sarima_predict(train_end,valid_origins)
    val_predictions["SARIMA"]=sarima_val
else:
    sarima_val=None
    print("SARIMA désactivé : activer RUN_SARIMA pour lancer l’ajustement.")

if RUN_CALENDAR_ABLATION and neural_specs:
    best=min(neural_val,key=lambda k:score(valid_truth,neural_val[k])["MAE_kW"])
    spec=neural_specs[best]; rows=[]
    for use_cal in [False,True]:
        model=fit_net(spec["kind"],spec["lookback"],train_origins,xs_train,mean_train,std_train,use_calendar=use_cal)
        pred=predict_net(model,spec["lookback"],valid_origins,xs_train,mean_train,std_train,use_calendar=use_cal)
        rows.append({"calendrier":use_cal,**score(valid_truth,pred)})
    display(pd.DataFrame(rows).set_index("calendrier").round(4))
elif RUN_CALENDAR_ABLATION:
    print("Activer d’abord le benchmark LSTM/TCN.")

## 9 — Comparer sur validation et analyser les erreurs

Le modèle est choisi sur validation selon MAE sur tous les horizons. RMSE et MAPE restent complémentaires. Les pics sont définis par le 95e percentile de l’entraînement. Ne pas modifier les choix après avoir consulté le test.

In [ ]:
all_val=dict(val_predictions); all_val.update(neural_val)
validation_table=pd.DataFrame({k:score(valid_truth,v) for k,v in all_val.items()}).T.sort_values("MAE_kW")
display(validation_table.round(4))
selected=str(validation_table.index[0])
print("Choix validation :",selected)
selected_val=all_val[selected]
h_mae=np.mean(np.abs(valid_truth-selected_val),axis=0)
pd.Series(h_mae,index=np.arange(1,25),name="MAE kW").plot(marker="o",figsize=(10,4))
plt.title("MAE de validation par horizon"); plt.xlabel("Horizon (heures)"); plt.xticks(np.arange(1,25))
plt.tight_layout(); plt.show()
threshold=float(np.nanquantile(y[:train_end],.95))
peaks=valid_truth>=threshold
display(pd.DataFrame([
    {"segment":"Toutes",**score(valid_truth,selected_val)},
    {"segment":"Pics",**score(valid_truth[peaks],selected_val[peaks])}
]).set_index("segment").round(4))

## 10 — Test final, intervalles et compte rendu

Les choix d’architecture et d’historique sont figés à partir de la validation. Pour le test, les transformations et les poids sont réajustés avec les données disponibles jusqu’à sa borne. Les intervalles sont des quantiles absolus des résidus de validation par horizon ; ils fournissent une couverture empirique, pas une garantie.

In [ ]:
test_truth=truth(test_origins)
if selected in BASELINES:
    test_pred=baseline(selected,test_origins,fallback=float(np.nanmedian(y[:valid_end])))
elif selected.startswith(("LSTM","TCN")):
    spec=neural_specs[selected]
    xs_tv,mean_tv,std_tv=preprocess(valid_end)
    model=fit_net(spec["kind"],spec["lookback"],trainval_origins,xs_tv,mean_tv,std_tv)
    test_pred=predict_net(model,spec["lookback"],test_origins,xs_tv,mean_tv,std_tv)
elif selected=="SARIMA":
    test_pred=sarima_predict(valid_end,test_origins)

display(pd.DataFrame([score(test_truth,test_pred)],index=[selected]).round(4))
q90=np.quantile(np.abs(valid_truth-selected_val),.90,axis=0)
i=int(test_origins[0]); dates=time_index[i+1:i+HORIZON+1]
lo=np.maximum(0,test_pred[0]-q90); hi=test_pred[0]+q90
plt.figure(figsize=(12,5))
plt.plot(dates,test_truth[0],marker="o",label="Observé")
plt.plot(dates,test_pred[0],marker="o",label="Prévu")
plt.fill_between(dates,lo,hi,alpha=.2,label="Intervalle empirique 90 %")
plt.title(f"Prévision 24 h — origine {time_index[i]}")
plt.ylabel("Puissance moyenne (kW)"); plt.xticks(rotation=45); plt.legend()
plt.tight_layout(); plt.show()
coverage=np.mean((test_truth>=np.maximum(0,test_pred-q90))&(test_truth<=test_pred+q90))
print(f"Couverture empirique sur le test : {coverage:.1%}")

## Compte rendu et références

Compléter avec les résultats effectivement calculés, sans inventer de score.

- Cible, heure d’émission et horizon 24 h : …
- Période, taux de données manquantes et fréquence après agrégation : …
- Baseline de référence et MAE : …
- Résultats LSTM / TCN / SARIMA et historique retenu : …
- Impact du calendrier : …
- Horizon le plus difficile, erreur sur les pics et couverture empirique : …
- Limites : un foyer, période 2006–2010, météo et vacances scolaires absentes.

Pistes : apparier météo horaire ; tester des validations glissantes ; étudier N-BEATS ou LSTNet ; comparer plusieurs foyers ; évaluer le coût de sous-prévision et de sur-prévision.

- Lai et al. (2018), LSTNet : https://arxiv.org/abs/1703.07015
- Oreshkin et al. (2020), N-BEATS : https://arxiv.org/abs/1905.10437
- UCI Household Electric Power Consumption : https://archive.ics.uci.edu/ml/datasets/individual+household+electric+power+consumption